# MLflow 5 minute Tracking Quickstart

In [1]:
# Start MLflow in de terminal:
#  activate venv
#  mlflow server --port 5000    # production, multiple users
# or
#  mlflow ui                    # local use on laptop

In [1]:
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

## Set the MLflow Tracking URI

In [3]:
# NOTE: review the links mentioned above for guidance on connecting to a managed tracking server, such as the Databricks Managed MLflow

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

## Load training data and train a simple model

In [4]:
# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the model hyperparameters
params = {"solver": "lbfgs", "max_iter": 1000, "random_state": 8888}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

# Calculate accuracy as a target loss metric
accuracy = accuracy_score(y_test, y_pred)

## Define an MLflow Experiment

In [5]:
mlflow.set_experiment("MLflow Quickstart")

2026/10/14 08:51:05 INFO mlflow.tracking.fluent: Experiment with name 'MLflow Quickstart' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1791960665622, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791960665622, lifecycle_stage='active', name='MLflow Quickstart', tags={}, trace_location=None, workspace='default'>

## Log the model, hyperparameters, and loss metrics to MLflow.

In [6]:
with mlflow.start_run():
  mlflow.log_params(params)                                       # Log the hyperparameters
  mlflow.log_metric("accuracy", accuracy)                         # Log the metric
  mlflow.set_tag("Training Info", "LR model for iris data")       # Add tag
  signature = infer_signature(X_train, lr.predict(X_train))       # Detects input & output schema
  model_info = mlflow.sklearn.log_model(                          # Log the model
      sk_model=lr,
      name="iris_model",
      signature=signature,
      input_example=X_train,
      registered_model_name="tracking-quickstart",  # With registered_model_name: model also uploaded in registry
  )

Successfully registered model 'tracking-quickstart'.
2026/10/14 08:51:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 1
Created version '1' of model 'tracking-quickstart'.


🏃 View run resilient-kite-922 at: http://127.0.0.1:5000/#/experiments/1/runs/98c0b5e278204210be80106d22d6e67d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Load our saved model as a Python Function

In [7]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

## Use our model to predict the iris class type on a Pandas DataFrame

In [8]:
predictions = loaded_model.predict(X_test)

iris_feature_names = datasets.load_iris().feature_names

# Convert X_test validation feature data to a Pandas DataFrame
result = pd.DataFrame(X_test, columns=iris_feature_names)

# Add the actual classes to the DataFrame
result["actual_class"] = y_test

# Add the model predictions to the DataFrame
result["predicted_class"] = predictions

result[:4]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
0,6.1,2.8,4.7,1.2,1,1
1,5.7,3.8,1.7,0.3,0,0
2,7.7,2.6,6.9,2.3,2,2
3,6.0,2.9,4.5,1.5,1,1
